In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Configuración de estilo
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

In [2]:
# Cargar el dataset
df = pd.read_csv('incidencias.csv')

print("Forma del dataset:", df.shape)
print("\nPrimeras filas:")
display(df.head())

# Asegurarnos de que no hay nulos en los campos que nos importan
df = df.dropna(subset=['description', 'categoria', 'urgencia'])

FileNotFoundError: [Errno 2] No such file or directory: 'incidencias.csv'

In [ ]:
# Definir las variables de entrada y la variable objetivo
X_text = df['description'].astype(str)
X_cat = df[['categoria']].copy() # Añade 'canal' a esta lista si lo vas a usar como variable

# La urgencia la tratamos como un número continuo de tipo float
y = df['urgencia'].astype(float)

# 1. Vectorización de Texto (TF-IDF)
tfidf = TfidfVectorizer(max_features=1000, stop_words='english', ngram_range=(1, 2))
X_text_tfidf = tfidf.fit_transform(X_text).toarray()

# 2. Codificación de Variables Categóricas (One-Hot Encoding)
X_cat_encoded = pd.get_dummies(X_cat, columns=['categoria'], drop_first=True).values

# 3. Concatenar todas las características en una sola matriz
X = np.hstack((X_text_tfidf, X_cat_encoded))

# Dividir el conjunto de datos en Entrenamiento (80%) y Prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Dimensiones de entrenamiento: {X_train.shape}")
print(f"Dimensiones de prueba: {X_test.shape}")

In [ ]:
# Inicializar y entrenar el modelo de regresión
# Puedes probar también con GradientBoostingRegressor()
rf_regressor = RandomForestRegressor(n_estimators=100, random_state=42)
rf_regressor.fit(X_train, y_train)

# Hacer predicciones en el conjunto de prueba
y_pred = rf_regressor.predict(X_test)

In [ ]:
# Limitamos las predicciones para que no se salgan del rango lógico (1.0 - 5.0)
y_pred_clipped = np.clip(y_pred, 1.0, 5.0)

# 1. Valor exacto con 2 decimales
y_pred_decimal = np.round(y_pred_clipped, 2)

# 2. Valor redondeado para la clasificación (Categoría de 1 a 5)
y_pred_rounded = np.round(y_pred_clipped).astype(int)

# --- Evaluación de Regresión ---
print("--- MÉTRICAS DE REGRESIÓN ---")
print(f"Error Cuadrático Medio (MSE): {mean_squared_error(y_test, y_pred_decimal):.4f}")
print(f"Error Absoluto Medio (MAE): {mean_absolute_error(y_test, y_pred_decimal):.4f}")
print(f"R2 Score: {r2_score(y_test, y_pred_decimal):.4f}")

# --- Evaluación del Redondeo vs Realidad (como Clasificación) ---
from sklearn.metrics import accuracy_score, classification_report

print("\n--- MÉTRICAS DE CLASIFICACIÓN (Urgencia Redondeada) ---")
print(f"Accuracy Exacta (predicción redondeada == real): {accuracy_score(y_test, y_pred_rounded):.4f}")
print("\nReporte de Clasificación:\n", classification_report(y_test, y_pred_rounded))

In [ ]:
def predecir_urgencia(descripcion, categoria):
    """
    Toma una descripción y su categoría y predice el nivel de urgencia.
    Retorna el valor con dos decimales y el nivel de clasificación (1-5).
    """
    # 1. Transformar texto
    text_tf = tfidf.transform([descripcion]).toarray()

    # 2. Transformar categoría a One-Hot respetando las columnas del entrenamiento
    cat_df = pd.DataFrame({'categoria': [categoria]})
    cat_encoded = pd.get_dummies(cat_df)

    # Asegurar que tiene exactamente las mismas columnas categóricas que X_cat
    expected_cols = pd.get_dummies(df[['categoria']], drop_first=True).columns
    cat_encoded = cat_encoded.reindex(columns=expected_cols, fill_value=0).values

    # 3. Concatenar
    X_new = np.hstack((text_tf, cat_encoded))

    # 4. Predecir
    pred_raw = rf_regressor.predict(X_new)[0]

    # 5. Limitar y Redondear
    pred_clipped = np.clip(pred_raw, 1.0, 5.0)
    urgencia_exacta = round(pred_clipped, 2)
    urgencia_clase = int(round(pred_clipped))

    return urgencia_exacta, urgencia_clase

# --- PRUEBA DEL MODELO ---
test_cases = [
    {
        "desc": "Un envase de productos químicos peligrosos ha sido abandonado en un parque, a 50 metros del colegio.",
        "cat": "Limpieza"
    },
    {
        "desc": "Pintada en forma de rótulos obscenos en la pared de mi casa.",
        "cat": "Otros"
    }
]

print("="*60)
print("PRUEBAS DE INFERENCIA EN EL AYUNTAMIENTO")
print("="*60)

for case in test_cases:
    valor_exacto, nivel = predecir_urgencia(case['desc'], case['cat'])
    print(f"Descripción: {case['desc']}")
    print(f"Categoría: {case['cat']}")
    print(f"Urgencia Calculada (Interna): {valor_exacto} / 5.0")
    print(f"Nivel de Urgencia Final Asignado: {nivel}\n")